# Arxiv Call

In [10]:
import urllib, urllib.request
import arxiv
import json 
from arxivql import Query as Q
from datetime import date, datetime

In [ ]:
import pandas as pd
from datetime import date
import re
import arxiv
from itertools import islice

import json

def paper_to_dict(paper):
    return {
        "entry_id": paper.entry_id,
        "title": paper.title,
        "authors": [author.name for author in paper.authors],
        "summary": paper.summary,
        "published": paper.published.isoformat(),
        "updated": paper.updated.isoformat(),
        "primary_category": paper.primary_category,
        "categories": paper.categories,
        "comment": paper.comment,
        "journal_ref": paper.journal_ref,
        "doi": paper.doi,
        "pdf_url": paper.pdf_url,
        "links": [
            {
                "href": link.href,
                "title": link.title,
                "rel": link.rel,
                "content_type": link.content_type,
            }
            for link in paper.links
        ],
    }

columns = ["title", "authors", "venue", "published", "journal_ref", "comment", "doi", "arxiv_url", "pdf_url"]
paper_df = pd.DataFrame([], columns=columns)


ACCEPTANCE_PATTERN = re.compile(
    r"\baccepted\b|\bto appear\b|\bpublished\b|\bproceedings\b",
    flags=re.IGNORECASE,
)


def find_accepted_venue(paper):
    metadata = " ".join(
        field for field in [paper.journal_ref, paper.comment]
        if field
    )

    # A journal reference is stronger evidence of publication.
    has_publication_evidence = bool(paper.journal_ref)

    # Otherwise, require explicit acceptance language in the comment.
    if not has_publication_evidence:
        has_publication_evidence = bool(
            ACCEPTANCE_PATTERN.search(paper.comment or "")
        )

    if not has_publication_evidence:
        return None

    for venue, pattern in VENUE_PATTERNS.items():
        if re.search(pattern, metadata, flags=re.IGNORECASE):
            return venue

    return None

# -----------------------------
# 1. Build the RAG query
# -----------------------------

q_category = Q.category("cs.IR") | Q.category("cs.LG") | Q.category("cs.CL") | Q.category("cs.AI") | Q.category("stat.ML")
q_date = Q.submitted_date(date(2020, 1, 1), None)

q_rag = (
    Q.title("retrieval augmented generation")
    | Q.abstract("retrieval augmented generation")
    | Q.title("retrieval-augmented generation")
    | Q.title("Large Language Model")
    | Q.abstract("retrieval-augmented generation")
    | Q.abstract("retrieval augmented language model")
)

query = q_category & q_date & q_rag


# -----------------------------
# 2. Define accepted venues
# -----------------------------

VENUE_PATTERNS = {
    "ACL": r"\bACL\b|Annual Meeting of the Association for Computational Linguistics",
    "EMNLP": r"\bEMNLP\b|Empirical Methods in Natural Language Processing",
    "NAACL": r"\bNAACL\b|North American Chapter of the Association for Computational Linguistics",
    "EACL": r"\bEACL\b|European Chapter of the Association for Computational Linguistics",
    "COLING": r"\bCOLING\b|International Conference on Computational Linguistics",
    "SIGIR": r"\bSIGIR\b|Conference on Research and Development in Information Retrieval",
    "NeurIPS": r"\bNeurIPS\b|Neural Information Processing Systems",
    "ICML": r"\bICML\b|International Conference on Machine Learning",
    "ICLR": r"\bICLR\b|International Conference on Learning Representations",
    "KDD": r"\bKDD\b|Knowledge Discovery and Data Mining",
    "AAAI": r"\bAAAI\b|Association for the Advancement of Artificial Intelligence",
    "IJCAI": r"\bIJCAI\b|International Joint Conference on Artificial Intelligence",
    "TACL": r"\bTACL\b|Transactions of the Association for Computational Linguistics",
    "JMLR": r"\bJMLR\b|Journal of Machine Learning Research",
}


def find_venue(paper):
    metadata = " ".join(
        field for field in [
            paper.journal_ref,
            paper.comment,
        ]
        if field
    )

    for venue, pattern in VENUE_PATTERNS.items():
        if re.search(pattern, metadata, flags=re.IGNORECASE):
            return venue

    return None


target_count = 400
batch_size = 1000
candidate_limit = 20_000

search = arxiv.Search(
    query=str(query),
    max_results=candidate_limit,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)

client = arxiv.Client(
    page_size=100,
    delay_seconds=3,
    num_retries=3,
)

# Create this generator only once
results_generator = client.results(search)

accepted_papers = []
candidates_checked = 0

with open("data/raw/raw_arxiv_results.jsonl", "w", encoding="utf-8") as file:
    for paper in results_generator:
        candidates_checked += 1

        raw_record = paper_to_dict(paper)

        # Save every retrieved candidate
        file.write(json.dumps(raw_record, ensure_ascii=False) + "\n")


with open("data/raw/raw_arxiv_results.jsonl", "r", encoding="utf-8") as file:
    raw_results = [json.loads(line) for line in file]

accepted_papers = []
candidates_checked = 0


while len(paper_df) < target_count:
    # Continue from where the generator stopped previously
    batch = list(islice(raw_results, batch_size))

    # No more matching arXiv results
    if not batch:
        print("No more matching papers are available.")
        break

    for paper in batch:
        candidates_checked += 1
        venue = find_accepted_venue(paper)

        if venue is None:
            continue
        if paper.title in paper_df["title"].values:
            continue

        accepted_papers.append({
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "venue": venue,
            "published": paper.published.date(),
            "journal_ref": paper.journal_ref,
            "comment": paper.comment,
            "doi": paper.doi,
            "arxiv_url": paper.entry_id,
            "pdf_url": paper.pdf_url,
        })

        if len(paper_df) >= target_count:
            break

        if len(accepted_papers) % 10 == 0:
            for index, paper in enumerate(accepted_papers, start=1):
                paper["authors"] = ", ".join(paper["authors"])  # Convert list of authors to a comma-separated string

            paper_df = pd.concat([paper_df, pd.DataFrame(accepted_papers, columns=columns)], ignore_index=True)
            accepted_papers = []  # Clear the list after adding to the DataFrame

    print(
        f"Checked {candidates_checked:,} candidates — "
        f"found {len(paper_df):,}/{target_count} verified papers."
    )

print(f"Found {len(paper_df)} venue-accepted papers.\n")

for index, paper in enumerate(paper_df.values, start=1):
    print(f"{index}. {paper['title']}")
    print(f"   Venue: {paper['venue']}")
    print(f"   Published on arXiv: {paper['published']}")
    print(f"   Journal reference: {paper['journal_ref']}")
    print(f"   URL: {paper['arxiv_url']}")
    print()

Checked 940 candidates — found 450/400 verified papers.
Checked 956 candidates — found 450/400 verified papers.
Checked 972 candidates — found 450/400 verified papers.
Checked 974 candidates — found 450/400 verified papers.
Checked 1,007 candidates — found 450/400 verified papers.
Checked 1,011 candidates — found 450/400 verified papers.
Checked 1,016 candidates — found 450/400 verified papers.
Checked 1,029 candidates — found 450/400 verified papers.
No more matching papers are available.
Found 450 venue-accepted papers.



IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

# Checking the arxiv papers json

In [1]:
import pandas as pd

In [6]:
import json
import pandas as pd

with open(r"../data/processed/arxiv_papers.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(records)
len(df["arxiv_id"].unique())

300